## Qwen3-8B on SageMaker with vLLM

This documents the commands are compatable with powershell terminal. If you are using a different terminal, please adjust the commands accordingly.

In [ ]:
$env:AWS_REGION  = "eu-central-1"
$env:AWS_ACCOUNT = (aws sts get-caller-identity --query Account --output text)

Now we will create Sagemaker IAM execution role. This role will enable sagemaker to pull the model weights from S3 bucket and run inference on the model.

In [ ]:
# Write the trust policy JSON file (@'...'@ = Bash heredoc equivalent)
@'
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": { "Service": "sagemaker.amazonaws.com" },
    "Action": "sts:AssumeRole"
  }]
}
'@ | Out-File -FilePath trust-policy.json -Encoding utf8
 

# Create the role
aws iam create-role `
  --role-name SageMakerVLLMRole `
  --assume-role-policy-document file://trust-policy.json
 
# Attach required policies
aws iam attach-role-policy `
  --role-name SageMakerVLLMRole `
  --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
 
aws iam attach-role-policy `
  --role-name SageMakerVLLMRole `
  --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess
 
aws iam attach-role-policy `
  --role-name SageMakerVLLMRole `
  --policy-arn arn:aws:iam::aws:policy/CloudWatchLogsFullAccess
 
# Save the role ARN
$env:ROLE_ARN = (aws iam get-role --role-name SageMakerVLLMRole `
  --query Role.Arn --output text)
Write-Host "Role ARN: $($env:ROLE_ARN)"



Now we are downloading the open source model Qwen3-8B from HuggingFace and uploading it to the S3 bucket.

In [ ]:
# Set your bucket name (must be globally unique)
$env:MODEL_BUCKET = "qwen3-weights"   

# Create bucket — eu-central-1 requires --create-bucket-configuration
aws s3api create-bucket `
  --bucket $env:MODEL_BUCKET `
  --region $env:AWS_REGION `
  --create-bucket-configuration LocationConstraint=$env:AWS_REGION
 
# Block all public access
aws s3api put-public-access-block `
  --bucket $env:MODEL_BUCKET `
  --public-access-block-configuration `
    "BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true"
 
# Download model weights from Hugging Face (~16 GB, 10-30 min)
pip install huggingface_hub
huggingface-cli download Qwen/Qwen3-8B `
  --local-dir ./qwen3-8b `
  --exclude "*.md" "*.txt"


#OR
pip install hf
hf download Qwen/Qwen3-8B --local-dir ./qwen3-8b

 
# Sync to S3 (STANDARD_IA = ~45% storage cost saving)
aws s3 sync ./qwen3-8b s3://$($env:MODEL_BUCKET)/Qwen3-8B/ `
  --storage-class STANDARD_IA


In [ ]:
4. Create the S3 Bucket

Create the S3 bucket in eu-central-1:

aws s3api create-bucket `
  --bucket qwen3-weights-eu-600865702484 `
  --region eu-central-1 `
  --create-bucket-configuration LocationConstraint=eu-central-1


Why?

The S3 bucket stores the Qwen3-8B model files.

Because the deployment is entirely in eu-central-1, we explicitly create the bucket in that region.

Important: S3 bucket names are globally unique. If qwen3-weights-eu-600865702484 is already taken in your AWS account or globally, choose another unique bucket name and use that same name throughout the guide.

4.1 Block Public Access

The model should not be publicly accessible.

Run:

aws s3api put-public-access-block `
  --bucket qwen3-weights-eu-600865702484 `
  --public-access-block-configuration "BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true" `
  --region eu-central-1


  
Why?

The Qwen3-8B model is a private deployment artifact.

SageMaker will access it using the SageMaker IAM execution role, not through public S3 access.

The four settings prevent accidental public access through:

BlockPublicAcls
IgnorePublicAcls
BlockPublicPolicy
RestrictPublicBuckets

Lets download the Qwen model from Huggingface 

pip install hf

hf download Qwen/Qwen3-8B --local-dir ./qwen3-8b


In [ ]:
aws s3 sync .\Qwen3-8B\ $env:MODEL_DATA --region $env:AWS_REGION

In [ ]:
#Create ECR Repository

aws ecr create-repository `
    --repository-name $env:ECR_REPO `
    --region $env:AWS_REGION

In [ ]:
#Log docker into ECR

aws ecr get-login-password --region $env:AWS_REGION |
    docker login --username AWS --password-stdin "$env:AWS_ACCOUNT_ID.dkr.ecr.$env:AWS_REGION.amazonaws.com"

In [ ]:
#Build the image

docker build `
    --platform linux/amd64 `
    --provenance=false `
    -t vllm-qwen3:sagemaker 

In [ ]:
#Tag the image for ECR

docker tag `
    vllm-qwen3:sagemaker `
    $env:IMAGE_URI

In [ ]:
#Push the image

docker push $env:IMAGE_URI

In [ ]:
#Create sagemaker model

aws sagemaker create-model `
    --model-name $env:MODEL_NAME `
    --execution-role-arn $env:ROLE_ARN `
    --primary-container "Image=$env:IMAGE_URI,Mode=SingleModel,ModelDataUrl=$env:MODEL_DATA,Environment={MODEL_DIR=/opt/ml/model,PORT=8080}" `
    --region $env:AWS_REGION

In [ ]:
Create endpoint configuration

In [ ]:


ENDPOINT_CONFIG_NAME = "qwen3-8b-config-fallback"

response = sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,

    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": "qwen3-8b-vllm",

            # Number of instances required
            "InitialInstanceCount": 1,

            # SageMaker tries these in priority order
            "InstancePools": [
                {
                    "InstanceType": "ml.g5.xlarge",
                    "Priority": 1,
                },
                {
                    "InstanceType": "ml.g6.2xlarge",
                    "Priority": 2,
                },
            ],

            # Give SageMaker enough time to provision
            "VariantInstanceProvisionTimeoutInSeconds": 600,

            # Give the container enough time to start vLLM
            "ContainerStartupHealthCheckTimeoutInSeconds": 600,
        }
    ],
)

print(response["EndpointConfigArn"])

In [ ]:
#Create Endpoint
response = sm.create_endpoint(
    EndpointName="qwen3-8b-endpoint",
    EndpointConfigName="qwen3-8b-config-fallback",
)

print(response["EndpointArn"])

In [ ]:
#Test Endpoint

import json
import boto3

REGION = "eu-central-1"
ENDPOINT_NAME = "qwen3-8b-endpoint-v2"

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

payload = {
    "model": "qwen3-8b",
    "messages": [
        {
            "role": "user",
            "content": "Explain what a transformer model is in one sentence."
        }
    ],
    "max_tokens": 100,
    "temperature": 0.7
}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read())

print(json.dumps(result, indent=2))

# Print just the generated answer
try:
    print("\n--- MODEL RESPONSE ---")
    print(result["choices"][0]["message"]["content"])
except (KeyError, IndexError):
    pass

In [ ]:
#END!

In [ ]:
# Create the trust policy file
@'
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": { "Service": "sagemaker.amazonaws.com" },
    "Action": "sts:AssumeRole"
  }]
}
'@ | Set-Content -Path trust-policy.json

# Create the role
aws iam create-role `
  --role-name SageMakerVLLMRole `
  --assume-role-policy-document file://trust-policy.json

# Attach required policies
aws iam attach-role-policy `
  --role-name SageMakerVLLMRole `
  --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess

aws iam attach-role-policy `
  --role-name SageMakerVLLMRole `
  --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess

aws iam attach-role-policy `
  --role-name SageMakerVLLMRole `
  --policy-arn arn:aws:iam::aws:policy/CloudWatchLogsFullAccess

# Save the role ARN
$ROLE_ARN = aws iam get-role `
  --role-name SageMakerVLLMRole `
  --query "Role.Arn" `
  --output text

Write-Host "Role ARN: $ROLE_ARN"

In [ ]:
# Create the bucket
$MODEL_BUCKET = "your-org-qwen3-weights"   # change this
aws s3 mb "s3://$MODEL_BUCKET" --region $AWS_REGION

# Block public access
aws s3api put-public-access-block `
  --bucket $MODEL_BUCKET `
  --public-access-block-configuration "BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true"

# Download model from Hugging Face (~16 GB, takes 10-30 min)
pip install hf

hf download Qwen/Qwen3-8B --local-dir ./qwen3-8b

# Upload to S3 (STANDARD_IA saves ~45% on storage costs)
aws s3 sync ./qwen3-8b "s3://$MODEL_BUCKET/Qwen3-8B/" `
  --storage-class STANDARD_IA

# Verify upload
aws s3 ls "s3://$MODEL_BUCKET/Qwen3-8B/" --human-readable --summarize

In [ ]:
Build command

Because SageMaker requires a Linux-compatible container, build the image for Linux AMD64:

docker build `
  --platform linux/amd64 `
  --provenance=false `
  --sbom=false `
  -t vllm-qwen3:latest .

In [ ]:
5. Create the ECR repository

Create the repository in the same region as SageMaker:

aws ecr create-repository `
  --repository-name vllm-qwen3 `
  --region eu-central-1

Why?

Amazon ECR is used to store the Docker image so that SageMaker can pull the image when creating the model.

For a clean deployment, the ECR repository should be in the same AWS region as SageMaker.

In [ ]:
6. Authenticate Docker with ECR

Run:

aws ecr get-login-password `
  --region eu-central-1 |
docker login `
  --username AWS `
  --password-stdin 600865702484.dkr.ecr.eu-central-1.amazonaws.com


Why?

Docker needs permission to push the locally built image into the private ECR repository.

In [ ]:
docker inspect vllm-qwen3:latest --format '{{.Os}}/{{.Architecture}}'

In [ ]:
Push image to ECR

$env:AWS_REGION = "us-east-1"
$env:AWS_ACCOUNT = aws sts get-caller-identity --query Account --output text
$env:ECR_URI = "$env:AWS_ACCOUNT.dkr.ecr.$env:AWS_REGION.amazonaws.com/vllm-qwen3"

In [ ]:
Create repository

aws ecr create-repository `
  --repository-name vllm-qwen3 `
  --region $env:AWS_REGION

In [ ]:
login

aws ecr get-login-password --region $env:AWS_REGION |
  docker login --username AWS --password-stdin "$env:AWS_ACCOUNT.dkr.ecr.$env:AWS_REGION.amazonaws.com"

In [ ]:
Tag

docker tag vllm-qwen3:latest "$env:ECR_URI`:latest"

In [ ]:
Push 

docker push "$env:ECR_URI`:latest"

In [ ]:
Verify

aws ecr batch-get-image `
  --repository-name vllm-qwen3 `
  --image-ids imageTag=latest `
  --region $env:AWS_REGION `
  --query "images[0].imageManifestMediaType" `
  --output text

In [ ]:
import boto3

REGION = "eu-central-1"

sm = boto3.client("sagemaker", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

ACCOUNT = sts.get_caller_identity()["Account"]

ECR_URI = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/vllm-qwen3:latest"
MODEL_DATA = "s3://qwen3-weights-eu-600865702484/Qwen3-8B/"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/SageMakerVLLMRole"

print("Region:", sm.meta.region_name)
print("Account:", ACCOUNT)
print("ECR:", ECR_URI)
print("S3:", MODEL_DATA)
print("Role:", ROLE_ARN)

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name="eu-central-1")

sm.delete_endpoint(
    EndpointName="qwen3-8b-endpoint"
)

print("Failed endpoint deletion requested.")

In [ ]:
import time

while True:
    try:
        sm.describe_endpoint(
            EndpointName="qwen3-8b-endpoint"
        )
        print("Endpoint still exists...")
        time.sleep(10)

    except sm.exceptions.ClientError:
        print("Endpoint deleted.")
        break

In [1]:
response = sm.create_endpoint(
    EndpointName="qwen3-8b-endpoint",
    EndpointConfigName="qwen3-8b-config",
)


print("Endpoint creation started.")
print(response["EndpointArn"])

NameError: name 'sm' is not defined

In [ ]:
import time

while True:
    response = sm.describe_endpoint(
        EndpointName="qwen3-8b-endpoint"
    )

    status = response["EndpointStatus"]
    print("Endpoint status:", status)

    if status == "InService":
        print("✅ Endpoint is ready!")
        break

    if status in ["Failed", "OutOfService"]:
        print("❌ Endpoint failed.")
        print(response.get("FailureReason"))
        break

    time.sleep(30)

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name="eu-central-1")

sm.delete_endpoint(
    EndpointName="qwen3-8b-endpoint"
)

print("Endpoint deletion requested.")

In [3]:
import boto3

sm = boto3.client("sagemaker", region_name="eu-central-1")

response = sm.describe_endpoint(
    EndpointName="qwen3-8b-endpoint"
)

print("Status:", response["EndpointStatus"])
print("Failure:", response.get("FailureReason"))

ClientError: An error occurred (ValidationException) when calling the DescribeEndpoint operation: Could not find endpoint "qwen3-8b-endpoint".

In [4]:
import boto3

sm = boto3.client("sagemaker", region_name="eu-central-1")

# Check the existing configuration
response = sm.describe_endpoint_config(
    EndpointConfigName="qwen3-8b-config"
)

print(response["ProductionVariants"][0]["InstanceType"])

ml.g5.xlarge


In [20]:
ENDPOINT_CONFIG_NAME = "qwen3-8b-config-fallback"

response = sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,

    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": "qwen3-8b-vllm",

            # Number of instances required
            "InitialInstanceCount": 1,

            # SageMaker tries these in priority order
            "InstancePools": [
                {
                    "InstanceType": "ml.g5.xlarge",
                    "Priority": 1,
                },
                {
                    "InstanceType": "ml.g6.2xlarge",
                    "Priority": 2,
                },
            ],

            # Give SageMaker enough time to provision
            "VariantInstanceProvisionTimeoutInSeconds": 600,

            # Give the container enough time to start vLLM
            "ContainerStartupHealthCheckTimeoutInSeconds": 600,
        }
    ],
)

print(response["EndpointConfigArn"])

ClientError: An error occurred (ValidationException) when calling the CreateEndpointConfig operation: Cannot create already existing endpoint configuration "arn:aws:sagemaker:eu-central-1:600865702484:endpoint-config/qwen3-8b-config-fallback".

In [22]:
response = sm.describe_endpoint_config(
    EndpointConfigName="qwen3-8b-config-fallback"
)

variant = response["ProductionVariants"][0]

print("Model:", variant["ModelName"])
print("Initial instances:", variant["InitialInstanceCount"])
print("Instance pools:")

for pool in variant["InstancePools"]:
    print(
        f"  Priority {pool['Priority']}: "
        f"{pool['InstanceType']}"
    )

Model: qwen3-8b-vllm
Initial instances: 1
Instance pools:
  Priority 1: ml.g5.xlarge
  Priority 2: ml.g6.2xlarge


In [23]:
response = sm.create_endpoint(
    EndpointName="qwen3-8b-endpoint",
    EndpointConfigName="qwen3-8b-config-fallback",
)

print(response["EndpointArn"])

arn:aws:sagemaker:eu-central-1:600865702484:endpoint/qwen3-8b-endpoint


In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name="eu-central-1")

MODEL_NAME = "qwen3-8b-vllm-v2"

ROLE_ARN = "arn:aws:iam::600865702484:role/SageMakerVLLMRole"

IMAGE_URI = (
    "600865702484.dkr.ecr.eu-central-1.amazonaws.com/"
    "vllm-qwen3:sagemaker"
)

MODEL_S3_URI = "s3://qwen3-weights-eu-600865702484/Qwen3-8B/"

response = sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={
        "Image": IMAGE_URI,

        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": MODEL_S3_URI,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },

        "Environment": {
            "MODEL_DIR": "/opt/ml/model",
            "PORT": "8080",
        },
    },
)

print(response["ModelArn"])


arn:aws:sagemaker:eu-central-1:600865702484:model/qwen3-8b-vllm-v2


In [14]:
import boto3

sm = boto3.client("sagemaker", region_name="eu-central-1")

MODEL_NAME = "qwen3-8b-vllm-v2"
ENDPOINT_CONFIG_NAME = "qwen3-8b-config-v2"

response = sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.g5.xlarge",
            "InitialVariantWeight": 1.0,

            # Qwen3 model download + container startup
            "ModelDataDownloadTimeoutInSeconds": 1800,
            "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        }
    ],
)

print(response["EndpointConfigArn"])

arn:aws:sagemaker:eu-central-1:600865702484:endpoint-config/qwen3-8b-config-v2


In [17]:
ENDPOINT_NAME = "qwen3-8b-endpoint-v2"

response = sm.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
)

print(response["EndpointArn"])

ClientError: An error occurred (ValidationException) when calling the CreateEndpoint operation: Cannot create already existing endpoint "arn:aws:sagemaker:eu-central-1:600865702484:endpoint/qwen3-8b-endpoint-v2".

In [16]:
aws sagemaker describe-endpoint `
  --endpoint-name qwen3-8b-endpoint-v2 `
  --region eu-central-1 `
  --query "EndpointStatus" `
  --output text

SyntaxError: invalid decimal literal (3183708488.py, line 2)

In [24]:
import json
import boto3

REGION = "eu-central-1"
ENDPOINT_NAME = "qwen3-8b-endpoint-v2"

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

payload = {
    "model": "qwen3-8b",
    "messages": [
        {
            "role": "user",
            "content": "Explain what a transformer model is in one sentence."
        }
    ],
    "max_tokens": 100,
    "temperature": 0.7
}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read())

print(json.dumps(result, indent=2))

# Print just the generated answer
try:
    print("\n--- MODEL RESPONSE ---")
    print(result["choices"][0]["message"]["content"])
except (KeyError, IndexError):
    pass

{
  "id": "chatcmpl-97433480ee0cdd01",
  "object": "chat.completion",
  "created": 1790276816,
  "model": "qwen3-8b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "<think>\nOkay, the user wants a one-sentence explanation of what a transformer model is. Let me start by recalling the basics. Transformers are a type of neural network architecture introduced in 2017. They're known for handling sequential data, like text, more efficiently than previous models like RNNs or LSTMs. The key innovation is the self-attention mechanism, which allows the model to weigh the importance of different words in a sentence. This enables the model to capture dependencies",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "reasoning": null
      },
      "logprobs": null,
      "finish_reason": "length",
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": nu